<a href="https://colab.research.google.com/github/ProductPriceTrackerOrg/data-science/blob/main/notebooks/GNN/01_sample_data_preperation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data Preperation for Build the Graph - GNN Data**

In [1]:
import hashlib
import json
import pandas as pd

In [2]:
# Load JSON
with open("/content/appleme_products.json", "r") as f:
    load1 = json.load(f)

In [3]:
# Load JSON
with open("/content/2025=08-31.json", "r") as f:
    load2 = json.load(f)

In [4]:
# Load JSON
with open("/content/cyberdeals_lk_scrape_optimized.json", "r") as f:
    load3 = json.load(f)

In [5]:
data_apple=load1['products']
data_cyber=load3['products']
data_life=load2

In [6]:
# brand = product_title[0] if not a number
for p in data_apple:
    if not p["brand"]:  # None or empty
        if p["product_title"]:
            p["brand"] = p["product_title"].split()[0] if not p["product_title"][0].isdigit() else None


In [7]:
for p in data_cyber:
    if not p["brand"]:  # None or empty
        if p["product_title"]:
            p["brand"] = p["product_title"].split()[0] if not p["product_title"][0].isdigit() else None

In [9]:
def generate_shop_product_id(source, pid):
    """Generate hex MD5 ID from shop + product id"""
    concat_str = f"{source}{pid}"
    return hashlib.md5(concat_str.encode("utf-8")).hexdigest()

In [10]:
# data = concatanate of data_apple, data_cyber, data_life
data = data_apple + data_cyber + data_life

# length of data
len(data)

25456

In [11]:
# product_df
product_df = pd.DataFrame([
    {
        "shop_product_id": generate_shop_product_id(
            p["metadata"]["source_website"], p["product_id_native"]
        ),
        "product_title_native": p["product_title"]
    }
    for p in data
])

In [12]:
# check there is any same shop_product_id is here
product_df['shop_product_id'].duplicated().sum()

np.int64(9)

In [13]:
# drop duplicates
product_df = product_df.drop_duplicates(subset=['shop_product_id'])

In [14]:
product_df.shape

(25447, 2)

In [15]:
# brand_df (unique brands)
brands = list({p["brand"] for p in data if p["brand"]})
brand_df = pd.DataFrame(brands, columns=["brand_native_name"])

In [16]:
brand_df.shape, brand_df.head()

((1017, 1),
   brand_native_name
 0           Burnout
 1            Selore
 2               CMF
 3              Cat6
 4             Forza)

In [17]:
# category_df
categories = set()
for p in data:
    categories.update(p["category_path"])

category_df = pd.DataFrame(
    [{"category_id": i, "category_name": c} for i, c in enumerate(categories, start=1)]
)

# category mapping for lookup
cat_map = {row["category_name"]: row["category_id"] for _, row in category_df.iterrows()}

In [18]:
category_df.shape

(641, 2)

In [19]:
# shop_df
shops = list({p["metadata"]["source_website"] for p in data})
shop_df = pd.DataFrame(
    [{"shop_id": i, "shop_name": s} for i, s in enumerate(shops, start=1)]
)

shop_map = {row["shop_name"]: row["shop_id"] for _, row in shop_df.iterrows()}

In [20]:
shop_df

,shop_id,shop_name
0,1,lifemobile.lk
1,2,appleme.lk
2,3,cyberdeals.lk


In [21]:
# product_brand_edges_df
product_brand_edges_df = pd.DataFrame([
    {
        "shop_product_id": generate_shop_product_id(
            p["metadata"]["source_website"], p["product_id_native"]
        ),
        "brand_native_name": p["brand"]
    }
    for p in data if p["brand"]
])

In [22]:
product_category_edges_df = pd.DataFrame([
    {
        "shop_product_id": generate_shop_product_id(
            p["metadata"]["source_website"], p["product_id_native"]
        ),
        "predicted_category_id": cat_map[p["category_path"][-1]] if p["category_path"] else None
    }
    for p in data
])

In [23]:
# product_shop_edges_df
product_shop_edges_df = pd.DataFrame([
    {
        "shop_product_id": generate_shop_product_id(
            p["metadata"]["source_website"], p["product_id_native"]
        ),
        "shop_id": shop_map[p["metadata"]["source_website"]]
    }
    for p in data
])

In [24]:
import os

# Create output folder if not exists
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

# Save all DataFrames to CSV
product_df.to_csv(os.path.join(output_dir, "product_df.csv"), index=False)
brand_df.to_csv(os.path.join(output_dir, "brand_df.csv"), index=False)
category_df.to_csv(os.path.join(output_dir, "category_df.csv"), index=False)
shop_df.to_csv(os.path.join(output_dir, "shop_df.csv"), index=False)
product_brand_edges_df.to_csv(os.path.join(output_dir, "product_brand_edges_df.csv"), index=False)
product_category_edges_df.to_csv(os.path.join(output_dir, "product_category_edges_df.csv"), index=False)
product_shop_edges_df.to_csv(os.path.join(output_dir, "product_shop_edges_df.csv"), index=False)